In [1]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [2]:
torch.manual_seed(1)

In [3]:
T = 16
b = 8
j = 8
lr = 0.001
epochs = 20
channels = 128

data_dir = os.path.expanduser('~/datasets/DVSGesture/')
out_dir = './logs'

In [4]:
device = 'cuda:0'

In [20]:
class DVSGestureNet(nn.Module):
    def __init__(self, channels=128, spiking_neuron: callable=None, is_seperable=False, kernel_size=3, **kwargs):
        super().__init__()

        conv = []
        for i in range(5):
            if conv.__len__() == 0:
                in_channels = 2
            else:
                in_channels = channels
            if is_seperable:
                conv.append(layer.Conv2d(in_channels, in_channels,
                                         kernel_size=kernel_size, groups=in_channels,
                                         padding=1, bias=False, step_mode='m')
                )
                conv.append(layer.Conv2d(in_channels, channels, kernel_size=1, step_mode='m'))
                conv.append(layer.BatchNorm2d(channels, step_mode='m'))
            else:
                conv.append(layer.Conv2d(in_channels, channels, kernel_size=kernel_size, 
                                         padding=1, bias=False, step_mode='m'))
                conv.append(layer.BatchNorm2d(channels, step_mode='m'))
                
            conv.append(spiking_neuron(**deepcopy(kwargs)))
            conv.append(layer.MaxPool2d(2, 2, step_mode='m'))


        self.conv_fc = nn.Sequential(
            *conv,

            layer.Flatten(step_mode='m'),
            layer.Dropout(0.5,  step_mode='m'),
            layer.Linear(channels * 4 * 4, 512, step_mode='m'),
            spiking_neuron(**deepcopy(kwargs)),

            layer.Dropout(0.5, step_mode='m'),
            layer.Linear(512, 110, step_mode='m'),
            spiking_neuron(**deepcopy(kwargs)),

            layer.VotingLayer(10, step_mode='m')
        )

    def forward(self, x: torch.Tensor):
        return self.conv_fc(x)


In [6]:
train_set = DVS128Gesture(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = DVS128Gesture(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.
The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.


In [7]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [8]:
scaler = amp.GradScaler()

In [9]:
def check_model(net):
    start_time = time.time()
    max_test_acc = -1
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, 11).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame).mean(0)
                    loss = F.mse_loss(out_fr, label_onehot)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame).mean(0)
                loss = F.mse_loss(out_fr, label_onehot)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, 11).float()
                out_fr = net(frame).mean(0)
                loss = F.mse_loss(out_fr, label_onehot)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    
    end_time = time.time()
    print(f'total time: {end_time - start_time}')

In [26]:
net = DVSGestureNet(
    channels=channels,
    spiking_neuron=neuron.LIFNode,
    surrogate_function=surrogate.ATan(),
    is_seperable=False,
    detach_reset=True
)

In [27]:
net.to(device)

DVSGestureNet(
  (conv_fc): Sequential(
    (0): Conv2d(2, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=m)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (2): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False, step_mode=m)
    (4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, step_mode=m)
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (6): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False, step_mode=m)
    (8): Con

In [28]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

In [29]:
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

In [30]:
check_model(net)

epoch = 0, train_loss = 0.0673, train_acc = 0.3937, test_loss = 0.0463, test_acc = 0.6146, max_test_acc = 0.6146
epoch = 1, train_loss = 0.0472, train_acc = 0.6199, test_loss = 0.0436, test_acc = 0.6285, max_test_acc = 0.6285
epoch = 2, train_loss = 0.0416, train_acc = 0.6412, test_loss = 0.0388, test_acc = 0.6840, max_test_acc = 0.6840
epoch = 3, train_loss = 0.0384, train_acc = 0.6752, test_loss = 0.0394, test_acc = 0.6528, max_test_acc = 0.6840
epoch = 4, train_loss = 0.0340, train_acc = 0.7228, test_loss = 0.0340, test_acc = 0.7326, max_test_acc = 0.7326
epoch = 5, train_loss = 0.0317, train_acc = 0.7364, test_loss = 0.0352, test_acc = 0.6875, max_test_acc = 0.7326
epoch = 6, train_loss = 0.0287, train_acc = 0.7730, test_loss = 0.0397, test_acc = 0.6632, max_test_acc = 0.7326
epoch = 7, train_loss = 0.0281, train_acc = 0.7747, test_loss = 0.0367, test_acc = 0.6840, max_test_acc = 0.7326
epoch = 8, train_loss = 0.0266, train_acc = 0.7755, test_loss = 0.0288, test_acc = 0.7812, max_t

In [31]:
net = DVSGestureNet(
    channels=channels,
    spiking_neuron=neuron.LIFNode,
    surrogate_function=surrogate.ATan(),
    is_seperable=True,
    detach_reset=True
)

In [32]:
net.to(device)

DVSGestureNet(
  (conv_fc): Sequential(
    (0): Conv2d(2, 2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=2, bias=False, step_mode=m)
    (1): Conv2d(2, 128, kernel_size=(1, 1), stride=(1, 1), step_mode=m)
    (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (3): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True)
    )
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False, step_mode=m)
    (5): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=128, bias=False, step_mode=m)
    (6): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), step_mode=m)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True, step_mode=m)
    (8): LIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=s, backend=torch, tau=2.0
      (

In [33]:
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

In [34]:
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

In [35]:
check_model(net)

epoch = 0, train_loss = 0.0676, train_acc = 0.3971, test_loss = 0.0472, test_acc = 0.6181, max_test_acc = 0.6181
epoch = 1, train_loss = 0.0458, train_acc = 0.6276, test_loss = 0.0413, test_acc = 0.6736, max_test_acc = 0.6736
epoch = 2, train_loss = 0.0387, train_acc = 0.6820, test_loss = 0.0359, test_acc = 0.6979, max_test_acc = 0.6979
epoch = 3, train_loss = 0.0355, train_acc = 0.7024, test_loss = 0.0343, test_acc = 0.7361, max_test_acc = 0.7361
epoch = 4, train_loss = 0.0316, train_acc = 0.7406, test_loss = 0.0356, test_acc = 0.7083, max_test_acc = 0.7361
epoch = 5, train_loss = 0.0297, train_acc = 0.7619, test_loss = 0.0311, test_acc = 0.7569, max_test_acc = 0.7569
epoch = 6, train_loss = 0.0278, train_acc = 0.7798, test_loss = 0.0308, test_acc = 0.7674, max_test_acc = 0.7674
epoch = 7, train_loss = 0.0258, train_acc = 0.7976, test_loss = 0.0294, test_acc = 0.7812, max_test_acc = 0.7812
epoch = 8, train_loss = 0.0247, train_acc = 0.8121, test_loss = 0.0276, test_acc = 0.8160, max_t